# Notebook 0.2 - Goals and assumptions of this calibration

## Goals of these notebooks

The goal of these notebooks is to calibrate 3 tree species for a study landscape in Quebec, Canada.

I limit things to 6 tree species as these notebooks are a proof of concept that will be used, if successful, for further calibration of more species in different Canadian landscapes.

The study landscape used here in Quebec, Canada, corresponds to a part of the territory of the Atikamekw first nation community of Manawan. This territory has been used before for previous LANDIS-II simulation, which will make comparisons to previous parameterization with Biomass Succession easier.

Here is a map showing the extent of the study landscape in space :

In [3]:
# Displays a map where the shapefile that defines the boundaries of the climate data we wanna use is
import geopandas as gpd
import folium
from folium.features import GeoJsonPopup, GeoJsonTooltip
from functionsForCalibration import *

# Load the shapefile
# INDICATE YOUR SHAPEFILE PATH HERE
studyLandscapeShapefilePath = "./ReferencesAndData/SpatialBoundaries/ManawanTerritoryExtent.shp"

# Computes its latitude and exports it to a JSON file - IMPORTANT, used in the other notebooks
compute_avg_latitude(studyLandscapeShapefilePath, './ReferencesAndData/SpatialBoundaries/studyLandscapeInformation.json')
print("✔ LATITUDE NOW INITIALIZED - EVERYTHING IS NOW IN ORDER")

# Replace 'path_to_shapefile.shp' with your actual shapefile path
gdf = gpd.read_file(studyLandscapeShapefilePath)

# Check the CRS (Coordinate Reference System) of the shapefile
# If not in WGS84 (EPSG:4326), reproject it
if gdf.crs != 'EPSG:4326':
    gdf = gdf.to_crs('EPSG:4326')

# Create a map centered on the mean of the shapefile bounds
center_lat = gdf.unary_union.centroid.y
center_lon = gdf.unary_union.centroid.x
m = folium.Map(location=[center_lat, center_lon], zoom_start=6)

# Add the GeoJSON data to the map with some styling
folium.GeoJson(
    gdf,
    name='Polygons',
    style_function=lambda x: {
        'fillColor': '#ebcb8b',
        'color': '#2e3440',
        'weight': 1,
        'fillOpacity': 0.4
    }
).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

# Alternatively, you can use this simpler approach which works in newer versions of JupyterLab
display(m)

Reprojecting from EPSG:32198 to EPSG:4326...
JSON written to ./ReferencesAndData/SpatialBoundaries/studyLandscapeInformation.json
✔ LATITUDE NOW INITIALIZED - EVERYTHING IS NOW IN ORDER


## Assumptions

*The text here is directly from the [PnET-Succession Calibration Tool](https://github.com/Klemet/PnET-Succession_Calibration_Tool).*

As per [Reese et al., 2024](https://doi.org/10.1139/cjfr-2024-0085) and other works, it is very important to document the assumptions (untested hypothesis) behind the choices done during the calibration.

Here, I catalog the different assumptions done throughout the calibration process. There is no need for you to learn or understand them all for now, as they will make sense while reading the rest of the notebooks.

### General assumptions

- (Derived from Reese et al., Appendix A) : A succession model built on ecophysiological first principles — and validated against historical empirical data — is assumed capable of making reliable projections under novel future conditions (e.g., climates or disturbance regimes with no historical precedent). Because the model is grounded in fundamental mechanisms rather than fitted to past patterns alone, its process-based structure is expected to remain valid even when environmental conditions move beyond the range of observations used during calibration and verification.

### Assumptions about notebook 3 (aquisition of initial parameters)

- Parameters from different geographic origins (e.g., measurements from trees outside and inside the study landscape) can be mixed without problematic interactions. However, the closer the empirical data or parameter estimates are to the study area, the more precise the calibration becomes, as fewer variant or phenotype behaviors are blended together.
- Water and temperature parameters, while not directly calibrated due to the lack of reliable empirical references as to their effects, can be based on previously published values through a statistical model relating to species' drought, waterlogging, and temperature tolerance. This is assumed sufficient to capture the variability of species' stress responses and their competitive interactions. This remains one of the weakest point of this calibration and of all PnET calibrations to date in my experience.
- The rules and suggestions from Eric Gustafson in the User guide (see tip number 29) are sufficient to generate reasonable initial parameter estimates that can later be refined. This assumption only applies when using the dedicated function used at the end of Notebook 3 to generate parameters for a tree species where non exists in the current litterature.

### Assumptions about notebook 4 (acquisition of calibration targets from empirical data and studies)

- NFI raster data, which represents species biomass within pixels of a given forest age, can be used through a space-for-time substitution to estimate species growth curves. The maximum biomass observed at a given age window across all pixels is assumed to represent the maximum growth achievable by that species for this given age window.
- Generalized Additive Models (GAMs) applied to NFI data allow estimation of species growth under monoculture conditions.
- Longevity values from Loehle (1988) are valid estimates for tree species in the study area, despite longevity varying with local growing conditions (especially temperature).
- Species tolerance values for shade, drought, waterlogging, and temperature (from Niinemets and Valladares 2006) may vary between variants or genotypes within a species, but the relative differences between species are assumed to remain consistent across landscapes. For example, if species A has a shade tolerance of 1 and species B has 3, species A will always be more shade-intolerant than B regardless of the landscape.
- Maximum LAI targets from Gustafson's calibration tips (see Table 1 in the tips) are representative of the differences of LAI between species of varying shade tolerances, and can thus serve as correct calibration targets. This is another weak part of the current calibration which could be improved.
- Published parameter value ranges from previous studies that used PnET-Succession represent appropriate bounds to respect during calibration.

### Assumptions about notebook 5 (generating the climate data stream used for the calibration in ideal conditions)

- Averaging the historical climate at the center of the study area produces a "mild climate" that represents the most ideal growing conditions — temperature and precipitation near their long-term averages — that the study landscape can offer (as it has no extreme weather events), especially when combined with a SILO (silt loam) soil.

### Assumptions about notebook 6 (calibrating the most sensitive parameters in ideal growth conditions)

- Calibrating species growth under "ideal conditions" (mild historical climate, monoculture, SILO soil), as recommended by the calibraiton tips of Eric Gustafson, isolates the effects of parameters that reduce photosynthesis (water, light, and temperature stress) and allows focused calibration of the most sensitive parameters (e.g., FolN) to determine the best growth achievable in the landscape. Since the mild climate stream is still derived from the study area's climate, this best growth remains constrained by temperature and precipitation, helping match the growth observed in the empirical NFI data. The resulting calibration is assumed to correctly attribute growth limitations between temperature/water stress and other sensitive parameters such as FolN and those piloting the LAI of the age cohorts.
- TOWood and TORoot are used to adjust biomass peak height during calibration, as they are the best parameters for this purpose with minimal influence on other age-cohort measurements (LAI and peak age in particular). This means TOWood and TORoot values are likely not ecologically realistic, but achieving correct growth is assumed more important than realistic wood and root turnover values. These parameters effectively serve as "trash bins" to absorb excess annual biomass production, enabling precise growth control. As such, when we have to change the parameters of a species outside of their bounds (see previous assumptions for bounds) to match the empirical data, TOWood and TORoot are two parameters that we will use in priority so that we can leave the other parameters inside their bounds as much as possible. Future calibration iterations or PnET-Succession updates may provide better levers for adjusting species growth. 
- The three most important aspects of a species growth to calibrate are: its growth peak height, its growth peak age, and its maximum LAI.
- The methodology developed by Brian Sturtevant's team for calibrating species longevity (described in detailed in Notebook 6) adequately represents the average decline trajectory of a tree species. Extreme cases of very long-lived trees — which cannot be reproduced by this approach since it assigns a cut-off death age to cohorts — are assumed unnecessary for capturing vegetation dynamics at the landscape scale.

### Assumptions about notebook 7 (assessing the competition between tree species and making tweaks)

- Identifying species that win or lose a competition of biomass when paired with another on the short-term (40 years) or long-term (200 years), combined with examining water and temperature influences on competitive ability for species at their range edges, is sufficient to detect critical competitive imbalances for landscape-level modeling. This is especially valid when corroborated by sources such as [Burns and Honkala (1990)](https://www.srs.fs.usda.gov/pubs/misc/ag_654/table_of_contents.htm). Given the extreme complexity of species competition in nature (and our prevalent inability to predict competition outcomes precisely in real forests), we assume that the model must be allowed to determine outcomes outside of these specific cases (too much losses, too much wins, not enough effect of water and temperature). In cases where we see that a species doesn't have the right competitive capabilities, we assume that simply adjusting its early growth by changing its growth peak age is enough to alter its competitive ability properly (without influencing other processes that should remain the same). 

### Assumptions about notebook 8 (calibrating MaxPest)

- Calibrating MaxPest on a small artificial landscape to stabilize the number of cohorts per cell below 12 after 50 years, while keeping it as high as possible, is sufficient to calibrate species dispersal and establishment capabilities. This is one of the weakest aspects of the calibration and may be improved in future iterations.